In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install wandb

In [3]:
import wandb
from kaggle_secrets import UserSecretsClient

api_key = UserSecretsClient().get_secret("WAND")
wandb.login(key=api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [4]:
import random, re

import torch
import torch.nn as nn

from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup

In [5]:
random.seed(33)
np.random.seed(33)
torch.manual_seed(33)

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(33)

In [6]:
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [8]:
OPTIONS = ["A", "B", "C", "D", "E"]

def split_prompt(prompt):
    text = str(prompt).strip()
    prefix, suffix = '', ''

    ci = text.find(': ')
    if ci != -1:
        prefix = text[:ci].strip()
        text = text[ci + 1:].strip()

    n_q = text.count('?')

    if n_q == 1:
        qi = text.index('?')
        suffix = text[qi + 1:].strip()
        text = text[:qi + 1].strip()

    elif n_q == 2:
        first_q = text.index('?')
        prefix = text[:first_q + 1].strip()

        text = text[first_q + 1:].strip()
        second_q = text.index('?')

        suffix = text[second_q + 1:].strip()
        text = text[:second_q + 1].strip()

    return (prefix or None), text, (suffix or None)


def clean(text, is_prompt=True):
    if(is_prompt):
        text = split_prompt(text)[1]
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# def tokenize(text, is_prompt=True):
#     return clean(text, is_prompt).split()


def ap_at_3(actual, predicted):
    for i, p in enumerate(predicted[:3]):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0


def map_at_3(actuals, predictions):
    return float(np.mean([ap_at_3(a, p) for a, p in zip(actuals, predictions)]))


def make_submission(ids, predictions):
    return pd.DataFrame({
        'id': ids,
        'prediction': [' '.join(p[:3]) for p in predictions]
    })

In [9]:
df['q_clean'] = df['prompt'].apply(clean)

for opt in OPTIONS:
    df[f'{opt}_clean'] = df[opt].apply(lambda text: clean(text, is_prompt=False))

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       2000 non-null   int64 
 1   prompt   2000 non-null   object
 2   A        2000 non-null   object
 3   B        2000 non-null   object
 4   C        2000 non-null   object
 5   D        2000 non-null   object
 6   E        2000 non-null   object
 7   answer   2000 non-null   object
 8   q_clean  2000 non-null   object
 9   A_clean  2000 non-null   object
 10  B_clean  2000 non-null   object
 11  C_clean  2000 non-null   object
 12  D_clean  2000 non-null   object
 13  E_clean  2000 non-null   object
dtypes: int64(1), object(13)
memory usage: 218.9+ KB


In [10]:
print("Empty questions:", (df['q_clean'].str.len() == 0).sum())

for opt in OPTIONS:
    empty_count = (df[f'{opt}_clean'].str.len() == 0).sum()
    print(opt, empty_count)

Empty questions: 0
A 0
B 0
C 0
D 0
E 0


In [11]:
unique_qs = df['q_clean'].unique()
(len(df), len(unique_qs), len(set(unique_qs)))

(2000, 252, 252)

In [12]:
NUM_FOLDS = 5

rng = np.random.default_rng(33)
shuffled_qs = unique_qs.copy()
rng.shuffle(shuffled_qs)

fold_qs = np.array_split(shuffled_qs, NUM_FOLDS)

for fold in range(NUM_FOLDS):
    print(f"Fold {fold}: {len(fold_qs[fold])} validation questions")

Fold 0: 51 validation questions
Fold 1: 51 validation questions
Fold 2: 50 validation questions
Fold 3: 50 validation questions
Fold 4: 50 validation questions


In [13]:
def create_pair_dataframe(source_df):
    pair_rows = []

    for row in source_df.itertuples(index=False):
        for opt in OPTIONS:
            pair_rows.append({
                'id': row.id,
                'option': opt,
                'question': row.q_clean,
                'option_text': getattr(row, f'{opt}_clean'),
                'label': int(row.answer == opt)
            })

    return pd.DataFrame(pair_rows)

In [14]:
fold0_val_qs = set(fold_qs[0])
fold0_train_qs = set(unique_qs) - fold0_val_qs

fold0_train_df = df[df['q_clean'].isin(fold0_train_qs)].reset_index(drop=True)
fold0_pairs = create_pair_dataframe(fold0_train_df)

print(fold0_pairs.head(10))

   id option                                           question  \
0   1      A  what is martin heidegger s view on the relatio...   
1   1      B  what is martin heidegger s view on the relatio...   
2   1      C  what is martin heidegger s view on the relatio...   
3   1      D  what is martin heidegger s view on the relatio...   
4   1      E  what is martin heidegger s view on the relatio...   
5   4      A  what is martin heidegger s view on the relatio...   
6   4      B  what is martin heidegger s view on the relatio...   
7   4      C  what is martin heidegger s view on the relatio...   
8   4      D  what is martin heidegger s view on the relatio...   
9   4      E  what is martin heidegger s view on the relatio...   

                                         option_text  label  
0  martin heidegger believes that humans exist wi...      0  
1  martin heidegger believes that humans do not e...      1  
2  martin heidegger does not believe in the exist...      0  
3  martin heid

In [ ]:
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_size = min(500, len(fold0_pairs))

sample_lengths = [
    len(tokenizer(row.question, row.option_text)['input_ids'])
    for row in fold0_pairs.sample(sample_size, random_state=33).itertuples(index=False)
]

pd.Series(sample_lengths).describe(percentiles=[0.50, 0.90, 0.95, 0.99])

In [16]:
def encode_groups(pair_df, tokenizer, max_len, has_labels=True):
    samples = []

    for question_id, group in pair_df.groupby('id', sort=False):
        group = group.sort_values('option')

        encoded = tokenizer(
            group['question'].tolist(),
            group['option_text'].tolist(),
            truncation=True,
            max_length=max_len,
            padding='max_length',
            return_tensors='pt'
        )

        sample = {
            'input_ids': encoded['input_ids'],
            'attention_mask': encoded['attention_mask'],
            'id': question_id
        }

        if has_labels:
            label_idx = group['label'].tolist().index(1)
            sample['label'] = torch.tensor(label_idx, dtype=torch.long)

        samples.append(sample)

    return samples

In [17]:
BATCH_SIZE = 32

In [18]:
def create_model():
    model = AutoModelForMultipleChoice.from_pretrained(
        MODEL_NAME,
        dtype=torch.float32
    )
    return model

In [19]:
LEARNING_RATE = 5e-5

In [20]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, total_samples = 0.0, 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        batch_size = input_ids.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples

In [21]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    predictions, actual_answers = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()

        batch_size = input_ids.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        labels_cpu = labels.cpu().numpy()

        for probability_row, label in zip(probabilities, labels_cpu):
            ranked_options = [OPTIONS[idx] for idx in np.argsort(-probability_row)]
            predictions.append(ranked_options[:3])
            actual_answers.append(OPTIONS[label])

    return {
        'loss': total_loss / total_samples,
        'map3': map_at_3(actual_answers, predictions),
        'predictions': predictions,
        'actual_answers': actual_answers
    }

In [22]:
EPOCHS = 15
PATIENCE = 5

def run_fold(fold, fold_qs):
    val_qs = set(fold_qs[fold])
    train_qs = set(unique_qs) - val_qs

    train_df = df[df['q_clean'].isin(train_qs)].reset_index(drop=True)
    val_df = df[df['q_clean'].isin(val_qs)].reset_index(drop=True)

    train_pairs = create_pair_dataframe(train_df)
    val_pairs = create_pair_dataframe(val_df)

    train_samples = encode_groups(train_pairs, tokenizer, MAX_LEN, has_labels=True)
    val_samples = encode_groups(val_pairs, tokenizer, MAX_LEN, has_labels=True)

    train_loader = DataLoader(train_samples, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_samples, batch_size=BATCH_SIZE*2, shuffle=False)

    model = create_model().to(device)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    best_map3 = -1.0
    best_epoch = 0
    epochs_without_improvement = 0

    history = []
    checkpoint_path = f'best_minilm_fold{fold}.pt'

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
        validation_result = evaluate(model, val_loader, device)

        val_loss = validation_result['loss']
        val_map3 = validation_result['map3']

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_map3': val_map3
        })

        print(f"Fold {fold}: Epoch {epoch} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val MAP@3: {val_map3:.4f}")
        
        run.log({
            "fold": fold,
            "epoch": epoch,
            "train/loss": train_loss,
            "val/loss": val_loss,
            "val/map3": val_map3
        })

        if val_map3 > best_map3:
            best_map3 = val_map3
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save(model.state_dict(), checkpoint_path)
            print(f"Fold {fold}, Best model saved: MAP@3={best_map3:.4f}")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f"Early stopping after epoch {epoch}.")
                break

    print(f"\nFold {fold} training complete")
    print(f"Fold {fold}, Best epoch: {best_epoch}")
    print(f"Fold {fold}, Best validation MAP@3: {best_map3}")

    run.log({
        "fold": fold,
        "fold/best_epoch": best_epoch,
        "fold/best_map3": best_map3
    })

    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    model.eval()

    best_validation_result = evaluate(model, val_loader, device)

    oof_ids = val_pairs['id'].unique().tolist()

    return {
        'fold': fold,
        'history': pd.DataFrame(history),
        'best_map3': best_map3,
        'model': model,
        'oof_ids': oof_ids,
        'oof_predictions': best_validation_result['predictions'],
        'oof_actuals': best_validation_result['actual_answers']
    }

In [23]:
run = wandb.init(
    project="23f2003391-t22026",
    name="minilm-oof",
    config={
        "model": MODEL_NAME,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "folds": NUM_FOLDS,
        "max_length": MAX_LEN,
    },
)

wandb: setting up run oshx9eq2
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260730_195819-oshx9eq2
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run minilm-oof
wandb: ⭐️ View project at https://wandb.ai/23f2003391-dl-genai-project/23f2003391-t22026
wandb: 🚀 View run at https://wandb.ai/23f2003391-dl-genai-project/23f2003391-t22026/runs/oshx9eq2


In [ ]:
fold_results = []

for fold in range(NUM_FOLDS):
    fold_results.append(run_fold(fold, fold_qs))

Fold 0: Epoch 1 | Train loss: 1.6063 | Val loss: 1.6053 | Val MAP@3: 0.4699  
Fold 0, Best model saved: MAP@3=0.4699  
Fold 0: Epoch 2 | Train loss: 1.2153 | Val loss: 1.6481 | Val MAP@3: 0.4956  
Fold 0, Best model saved: MAP@3=0.4956  
Fold 0: Epoch 3 | Train loss: 0.3602 | Val loss: 1.9916 | Val MAP@3: 0.5070  
Fold 0, Best model saved: MAP@3=0.5070  
Fold 0: Epoch 4 | Train loss: 0.1814 | Val loss: 1.7674 | Val MAP@3: 0.5217  
Fold 0, Best model saved: MAP@3=0.5217  
Fold 0: Epoch 5 | Train loss: 0.0865 | Val loss: 1.9565 | Val MAP@3: 0.5125  
Fold 0: Epoch 6 | Train loss: 0.0613 | Val loss: 1.8791 | Val MAP@3: 0.5213  
Fold 0: Epoch 7 | Train loss: 0.0447 | Val loss: 1.9933 | Val MAP@3: 0.5272  
Fold 0, Best model saved: MAP@3=0.5272  
Fold 0: Epoch 8 | Train loss: 0.0152 | Val loss: 2.1157 | Val MAP@3: 0.5543  
Fold 0, Best model saved: MAP@3=0.5543  
Fold 0: Epoch 9 | Train loss: 0.0091 | Val loss: 2.0614 | Val MAP@3: 0.5459  
Fold 0: Epoch 10 | Train loss: 0.0088 | Val loss: 2.0145 | Val MAP@3: 0.5448  
Fold 0: Epoch 11 | Train loss: 0.0038 | Val loss: 2.0484 | Val MAP@3: 0.5389  
Fold 0: Epoch 12 | Train loss: 0.0012 | Val loss: 2.0594 | Val MAP@3: 0.5294  
Fold 0: Epoch 13 | Train loss: 0.0009 | Val loss: 2.0929 | Val MAP@3: 0.5165  
Early stopping after epoch 13.

Fold 0 training complete  
Fold 0, Best epoch: 8  
Fold 0, Best validation MAP@3: 0.554331864904552  

Fold 1: Epoch 1 | Train loss: 1.6054 | Val loss: 1.6028 | Val MAP@3: 0.4424  
Fold 1, Best model saved: MAP@3=0.4424  
Fold 1: Epoch 2 | Train loss: 1.2282 | Val loss: 1.5837 | Val MAP@3: 0.4326  
Fold 1: Epoch 3 | Train loss: 0.4578 | Val loss: 1.7500 | Val MAP@3: 0.4894  
Fold 1, Best model saved: MAP@3=0.4894  
Fold 1: Epoch 4 | Train loss: 0.2231 | Val loss: 1.6981 | Val MAP@3: 0.4857  
Fold 1: Epoch 5 | Train loss: 0.1316 | Val loss: 2.0038 | Val MAP@3: 0.4783  
Fold 1: Epoch 6 | Train loss: 0.0807 | Val loss: 2.3458 | Val MAP@3: 0.4812  
Fold 1: Epoch 7 | Train loss: 0.0409 | Val loss: 2.3549 | Val MAP@3: 0.4681  
Fold 1: Epoch 8 | Train loss: 0.0278 | Val loss: 2.2274 | Val MAP@3: 0.4534  
Early stopping after epoch 8.

Fold 1 training complete  
Fold 1, Best epoch: 3  
Fold 1, Best validation MAP@3: 0.4893790849673202  

Fold 2: Epoch 1 | Train loss: 1.6064 | Val loss: 1.6084 | Val MAP@3: 0.3878  
Fold 2, Best model saved: MAP@3=0.3878  
Fold 2: Epoch 2 | Train loss: 1.2356 | Val loss: 1.5277 | Val MAP@3: 0.5122  
Fold 2, Best model saved: MAP@3=0.5122  
Fold 2: Epoch 3 | Train loss: 0.3850 | Val loss: 1.7262 | Val MAP@3: 0.5039  
Fold 2: Epoch 4 | Train loss: 0.1930 | Val loss: 1.9946 | Val MAP@3: 0.4404  
Fold 2: Epoch 5 | Train loss: 0.1294 | Val loss: 2.0738 | Val MAP@3: 0.4337  
Fold 2: Epoch 6 | Train loss: 0.0737 | Val loss: 1.8642 | Val MAP@3: 0.4588  
Fold 2: Epoch 7 | Train loss: 0.0432 | Val loss: 2.0176 | Val MAP@3: 0.4780  
Early stopping after epoch 7.

Fold 2 training complete  
Fold 2, Best epoch: 2  
Fold 2, Best validation MAP@3: 0.512156862745098  

Fold 3: Epoch 1 | Train loss: 1.6054 | Val loss: 1.6024 | Val MAP@3: 0.4726  
Fold 3, Best model saved: MAP@3=0.4726  
Fold 3: Epoch 2 | Train loss: 1.1770 | Val loss: 1.8474 | Val MAP@3: 0.4383  
Fold 3: Epoch 3 | Train loss: 0.3884 | Val loss: 1.7679 | Val MAP@3: 0.5288  
Fold 3, Best model saved: MAP@3=0.5288  
Fold 3: Epoch 4 | Train loss: 0.1698 | Val loss: 2.4470 | Val MAP@3: 0.4044  
Fold 3: Epoch 5 | Train loss: 0.0869 | Val loss: 2.3759 | Val MAP@3: 0.4596  
Fold 3: Epoch 6 | Train loss: 0.0387 | Val loss: 2.7332 | Val MAP@3: 0.4564  
Fold 3: Epoch 7 | Train loss: 0.0222 | Val loss: 2.7169 | Val MAP@3: 0.4369  
Fold 3: Epoch 8 | Train loss: 0.0131 | Val loss: 2.6251 | Val MAP@3: 0.4508  
Early stopping after epoch 8.

Fold 3 training complete  
Fold 3, Best epoch: 3  
Fold 3, Best validation MAP@3: 0.5287836583101206  

Fold 4: Epoch 1 | Train loss: 1.6043 | Val loss: 1.5999 | Val MAP@3: 0.5678  
Fold 4, Best model saved: MAP@3=0.5678  
Fold 4: Epoch 2 | Train loss: 1.1444 | Val loss: 1.3974 | Val MAP@3: 0.6073  
Fold 4, Best model saved: MAP@3=0.6073  
Fold 4: Epoch 3 | Train loss: 0.3466 | Val loss: 1.5497 | Val MAP@3: 0.5339  
Fold 4: Epoch 4 | Train loss: 0.1759 | Val loss: 1.7725 | Val MAP@3: 0.5466  
Fold 4: Epoch 5 | Train loss: 0.0960 | Val loss: 1.6849 | Val MAP@3: 0.5122  
Fold 4: Epoch 6 | Train loss: 0.0352 | Val loss: 1.9792 | Val MAP@3: 0.4896  
Fold 4: Epoch 7 | Train loss: 0.0191 | Val loss: 1.7549 | Val MAP@3: 0.4765  
Early stopping after epoch 7.

Fold 4 training complete  
Fold 4, Best epoch: 2  
Fold 4, Best validation MAP@3: 0.6073446327683616  

In [25]:
oof_predictions, oof_actuals = [], []

for result in fold_results:
    oof_predictions.extend(result['oof_predictions'])
    oof_actuals.extend(result['oof_actuals'])
    print(f"Fold {result['fold']} best validation MAP@3: {result['best_map3']:.4f}")

overall_map3 = map_at_3(oof_actuals, oof_predictions)
print("Overall OOF MAP@3:", overall_map3)

Fold 0 best validation MAP@3: 0.5543
Fold 1 best validation MAP@3: 0.4894
Fold 2 best validation MAP@3: 0.5122
Fold 3 best validation MAP@3: 0.5288
Fold 4 best validation MAP@3: 0.6073
Overall OOF MAP@3: 0.5369166666666666


In [26]:
run.summary["oof/map3"] = overall_map3
run.finish()

wandb: updating run metadata
wandb: uploading history steps 46-47, summary, console lines 126-137
wandb: 
wandb: Run history:
wandb:           epoch ▁▂▂▃▃▄▅▅▆▆▇▇█▂▂▃▃▄▅▅▁▂▂▃▃▄▁▂▂▃▃▄▅▅▁▂▂▃▃▅
wandb:            fold ▁▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▅▅▅▅▅▅▆▆▆▆▆▆▆▆██████
wandb: fold/best_epoch █▂▁▂▁
wandb:  fold/best_map3 ▅▁▂▃█
wandb:      train/loss █▆▃▂▁▁▁▁▁▁▁▁▁▆▃▂▂▁▁▁█▆▃▂▂▁█▆▃▂▁▁▁▁█▆▃▂▁▁
wandb:        val/loss ▂▂▄▃▄▄▄▅▄▄▄▄▅▂▃▃▄▆▆▅▂▂▃▄▅▃▂▃▃▆▆██▇▂▁▂▃▃▃
wandb:        val/map3 ▄▄▅▅▅▅▅▆▆▆▆▆▅▂▄▄▄▄▄▃▁▅▅▃▂▃▄▃▅▂▃▃▃▃▇█▆▆▅▄
wandb: 
wandb: Run summary:
wandb:           epoch 7
wandb:            fold 4
wandb: fold/best_epoch 2
wandb:  fold/best_map3 0.60734
wandb:        oof/map3 0.53692
wandb:      train/loss 0.01912
wandb:        val/loss 1.75493
wandb:        val/map3 0.47646
wandb: 
wandb: 🚀 View run minilm-oof at: https://wandb.ai/23f2003391-dl-genai-project/23f2003391-t22026/runs/oshx9eq2
wandb: ⭐️ View project at: https://wandb.ai/23f2003391-dl-genai-project/23f2003391-t22026
wandb: Synced 5 W&B file(s), 0

In [27]:
history_df = pd.concat(
    [result['history'].assign(fold=result['fold']) for result in fold_results],
    ignore_index=True
)
history_df

,epoch,train_loss,val_loss,val_map3,fold
0,1,1.606250,1.605289,0.469897,0
1,2,1.215329,1.648102,0.495595,0
2,3,0.360185,1.991650,0.506975,0
3,4,0.181365,1.767362,0.521659,0
4,5,0.086479,1.956501,0.512482,0
5,6,0.061252,1.879050,0.521292,0
6,7,0.044655,1.993349,0.527166,0
7,8,0.015164,2.115715,0.554332,0
8,9,0.009099,2.061357,0.545888,0
9,10,0.008849,2.014527,0.544787,0


In [28]:
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
test_df['q_clean'] = test_df['prompt'].apply(clean)

for opt in OPTIONS:
    test_df[f'{opt}_clean'] = test_df[opt].apply(lambda text: clean(text, is_prompt=False))

print(test_df.shape)
test_df.head()

(500, 13)


,id,prompt,A,B,C,D,E,q_clean,A_clean,B_clean,C_clean,D_clean,E_clean
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...",what is the relationship between the hamiltoni...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi...",what is the estimated redshift of ceers 93316 ...,approximately z 6 0 corresponding to 1 billion...,approximately z 16 7 corresponding to 235 8 mi...,approximately z 3 0 corresponding to 5 billion...,approximately z 10 0 corresponding to 13 billi...,approximately z 13 0 corresponding to 30 billi...
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...,what is the reason for the sun appearing sligh...,the sun appears yellowish due to a reflection ...,the longer wavelengths of light such as red an...,the sun appears yellowish due to the scatterin...,the sun emits a yellow light due to its own sp...,the atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,what is the significance of the redshift dista...,observations of the redshift distance relation...,observations of the redshift distance relation...,observations of the redshift distance relation...,observations of the redshift distance relation...,observations of the redshift distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,what is the landau lifshitz gilbert equation u...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...


In [29]:
def create_test_pair_dataframe(source_df):
    pair_rows = []

    for row in source_df.itertuples(index=False):
        for opt in OPTIONS:
            pair_rows.append({
                'id': int(row.id),
                'option': opt,
                'question': row.q_clean,
                'option_text': getattr(row, f'{opt}_clean')
            })

    return pd.DataFrame(pair_rows)

In [30]:
test_pairs = create_test_pair_dataframe(test_df)

(len(test_df), len(test_pairs))

(500, 2500)

In [31]:
@torch.no_grad()
def predict_test(model, loader, device):
    model.eval()

    ids_list, probabilities_list = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        logits = model(input_ids, attention_mask).logits
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()

        ids_cpu = batch['id'].cpu().numpy()

        ids_list.extend(ids_cpu.tolist())
        probabilities_list.extend(probabilities)

    return ids_list, probabilities_list

In [32]:
fold_probabilities = []

for result in fold_results:
    test_samples = encode_groups(test_pairs, tokenizer, MAX_LEN, has_labels=False)
    test_loader = DataLoader(test_samples, batch_size=BATCH_SIZE*2, shuffle=False)

    test_ids, test_probs = predict_test(result['model'], test_loader, device)

    fold_probabilities.append(np.array(test_probs))

ensemble_probabilities = np.mean(fold_probabilities, axis=0)

In [33]:
test_predictions = [
    [OPTIONS[idx] for idx in np.argsort(-probability_row)][:3]
    for probability_row in ensemble_probabilities
]

submission = make_submission(test_ids, test_predictions)
submission.to_csv('submission.csv', index=False)

submission.head(5)

,id,prediction
0,1,A C B
1,2,B E A
2,3,B D E
3,4,E C A
4,5,C A E
